In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP,SELECT
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import *

from network.layer import Layer
from scipy.optimize import curve_fit
import random

In [ ]:
# chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
# chip.set_device_cfg(deviceType=0,IsNew32=False)
# chip.adc.set_gap(adc_cs_gap=90,adc_first_gap=20,adc_last_gap=10)
# chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
# chip.clk_manager.set_cyc(10, 10,delay3=50)
# chip.add_compiler("../compiler/code/")

In [ ]:
chip=CHIP(PS(host="192.168.1.11", port = 7, debug=0),init=True)
# deviceType参数：0为ReRAM，1为ECRAM
# IsNew32参数：False为v1版本，True为v2版本
chip.set_device_cfg(deviceType=0,IsNew32=True)
chip.adc.set_sample_times(adc_sample_times=16)
chip.adc.set_gap(adc_cs_gap=100,adc_first_gap=1000,adc_last_gap=40)
chip.adc.set_gain_resistor(big_resistance=10e3,small_resistance=200)
chip.clk_manager.set_cyc(10, 10,delay3=0)
chip.add_compiler("./compiler/code/")
# chip.compensation.initop("../chip_data/chip6_/")

In [ ]:
# chip.setting.ins_ram_length = 

In [ ]:
crossbar = np.ones((256,256))
v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=False,split_type=1,row_type=0,col_type=0)
plot_cond(c,vmax=1000)

In [ ]:
ans = []
ans2 = []
for i in range(200):
    num = random.randint(5, 20)
    start = 0#random.randint(10, 200)
    v,c,r = chip.read4(crossbar=None,row_index=[i for i in range(start,start+num)],col_index=[101],read_voltage=0.1,tg=5,gain=1,sub_base=True,from_row=True,split_type=3,row_type=0,col_type=0)
    ans.append(c[101])
    v,c,r = chip.read4(crossbar=None,row_index=[i for i in range(start,start+num)],col_index=[101],read_voltage=0.1,tg=5,gain=3,sub_base=True,from_row=True,split_type=3,row_type=0,col_type=0)
    ans2.append(c[101])

In [ ]:
params, covariance = curve_fit(lambda x, mult: x * mult, np.array(ans2),np.array(ans), p0=[0.94],bounds=([0], [2]))
print(params)


In [ ]:
ans = np.array(ans)
ans2 = np.array(ans2)
plt.plot(ans,label="gain1")
plt.plot(ans2*0.93801281,label="gain3")
plt.legend()
plt.show()

# plt.plot(ans,label="gain1")
plt.plot(np.array(ans)/np.array(ans2),label="gain1/gain3")
plt.ylim((0.93,0.95))
plt.legend()
plt.show()

In [ ]:
ans = np.array(ans)
ans2 = np.array(ans2)
plt.plot(ans,label="gain1")
plt.plot(ans2*0.93801281,label="gain3")
plt.legend()
plt.show()

# plt.plot(ans,label="gain1")
plt.plot(np.array(ans)/np.array(ans2),label="gain1/gain3")
plt.ylim((0.93,0.95))
plt.legend()
plt.show()

In [ ]:

crossbar = np.ones((50,50))
for gain in [3,1,2,0]:
    ans = np.zeros((50,50,50))
    ansv = np.zeros((50,50,50))
    for i in range(50):
        v,c,r = chip.read4(crossbar=crossbar,row_index=None,col_index=None,read_voltage=0.1,tg=5,gain=gain,sub_base=True,from_row=False,split_type=0,row_type=0,col_type=0)
        ans[i,:,:]=c[:50,:50]
        ansv[i,:,:]=v[:50,:50]
    np.save(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy",ans)
    np.save(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy",ansv)

In [ ]:
gain = 3
ans_3 = np.load(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy")
ansv_3 = np.load(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy")


gain = 1
ans_1 = np.load(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy")
ansv_1 = np.load(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy")


gain = 2
ans_2 = np.load(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy")
ansv_2 = np.load(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy")


gain = 0
ans_0 = np.load(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_c.npy")
ansv_0 = np.load(f"./data/adc/pcb203/gain/gain={gain}_first_gap={2}_from_row={False}_sub_base={True}_v.npy")


# mean_matrix = np.mean(matrices, axis=0)[:20,:180]
ans = [np.mean(ans_0, axis=0),np.mean(ans_1, axis=0),np.mean(ans_2, axis=0),np.mean(ans_3, axis=0)]

In [ ]:
for i in range(1,2):
    plot_cond((ans[3]-ans[i])/ans[3],vmax = 2/15)
    plot_cond(ans[i])
    plt.hist((ans[3]-ans[1])/ans[i])
    # plt.title(f"gian={i}_read")
    plt.show()
    # plt.hist((ans[3]-ans[i])/ans[3])
    # plt.title(f"gian={3} - gain={i}")
    # plt.show()

In [ ]:
for i in range(4):
    plt.hist(ans[i])
    plt.title(f"gian={i}_read")
    plt.show()
    plt.hist((ans[3]-ans[i])/ans[3])
    plt.title(f"gian={3} - gain={i}")
    plt.show()
    
# plt.hist(ans[1]-ans[3])
# plt.show()
# plot_cond(ans[3]-ans[1],vmax=200)
# plot_cond(ans[1])